# SCRAP DATA 

test some techno 

In [2]:
import requests
from bs4 import BeautifulSoup

data = []
page_tabe = [1, 2, 3]
class_ = "h5"

for page in page_tabe:
    url = f"https://www.guidedeschampignons.com/tous-les-champignons/page/{page}"

    # 1. Télécharger la page
    response = requests.get(url)
    response.raise_for_status()

    print(f"--- Page {page} ---")
    print("status:", response.status_code)
    print("html length:", len(response.text))

    # 2. Parser le HTML
    soup = BeautifulSoup(response.text, "html.parser")

    # 3. Récupérer toutes les div avec la classe donnée
    divs = soup.find_all(class_=class_)

    # 4. Pour chaque div, récupérer les <a> qu'elle contient
    for div in divs:
        links = div.find_all("a")
        for a in links:
            data.append({
                "text": a.get_text(strip=True),
                "href": a.get("href")
            })
            print(a.get_text(strip=True), "->", a.get("href"))

print(len(data))


--- Page 1 ---
status: 200
html length: 1695416
AGARIC AUGUSTE -> https://www.guidedeschampignons.com/produit/agaric-auguste/
AGARIC DES JACHÈRES -> https://www.guidedeschampignons.com/produit/agaric-des-jacheres/
AGARIC JAUNISSANT -> https://www.guidedeschampignons.com/produit/agaric-jaunissant-agaricus-xanthoderma-champignon-toxique/
AGARIC SYLVICOLE -> https://www.guidedeschampignons.com/produit/agaric-sylvicole-bon-comestible/
AMANITE CITRINE -> https://www.guidedeschampignons.com/produit/amanite-citrine-amanita-citrina-champignon-a-rejeter/
AMANITE ÉPAISSE -> https://www.guidedeschampignons.com/produit/amanite-epaisse-amanita-excelsa-var-spissa-comestible-mediocre/
AMANITE FAUVE -> https://www.guidedeschampignons.com/produit/amanite-fauve-amanita-fulva-comestible-bien-cuit/
AMANITE JONQUILLE -> https://www.guidedeschampignons.com/produit/amanite-jonquille-amanita-junquillea-champignon-toxique/
AMANITE OVOÏDE -> https://www.guidedeschampignons.com/produit/amanite-ovoide-amanita-ovo

In [3]:
import time
import base64
import pandas as pd

CHAMPS = ["CHAPEAU", "PORES", "LAMELLES", "PIED", "CHAIR", "ODEUR", "SAVEUR", "HABITAT", "SAISON", "CONFUSION"]

def get_image_base64(url):
    if not url:
        return None
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    mime = response.headers.get("Content-Type", "image/jpeg").split(";")[0]
    b64 = base64.b64encode(response.content).decode("utf-8")
    return f"data:{mime};base64,{b64}"

def scrape_champignon(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    result = {"url": url}

    # Nom
    h1 = soup.find("h1", class_="product_title")
    result["nom"] = h1.get_text(strip=True) if h1 else None

    # Image principale
    gallery = soup.find("div", class_="woocommerce-product-gallery__image-first")
    image_url = None
    if gallery:
        a = gallery.find("a")
        image_url = a.get("href") if a else None
    result["image_url"] = image_url
    result["image_base64"] = get_image_base64(image_url)

    desc = soup.find("div", class_="woocommerce-product-details__short-description")
    if desc:
        paragraphs = desc.find_all("p")

        # Nom scientifique : 1er paragraphe, 1er <span>
        result["nom_scientifique"] = None
        if paragraphs:
            span = paragraphs[0].find("span")
            result["nom_scientifique"] = span.get_text(strip=True) if span else None

        # Statut : 2ème paragraphe, texte après l'image
        result["statut"] = None
        if len(paragraphs) > 1:
            p2 = paragraphs[1]
            for img in p2.find_all("img"):
                img.decompose()
            result["statut"] = " ".join(p2.get_text(separator=" ", strip=True).split())

        # Caractéristiques clés
        char_data = {c.lower(): None for c in CHAMPS}
        for p in paragraphs:
            strong = p.find("strong")
            if not strong:
                continue
            key = strong.get_text(strip=True).upper().rstrip(" :")
            if key in CHAMPS:
                full = p.get_text(separator=" ", strip=True)
                val = full[len(key):].lstrip(" :").strip()
                if key == "CONFUSION":
                    liens = [a.get_text(strip=True) for a in p.find_all("a")]
                    char_data["confusion"] = ", ".join(liens) if liens else val
                else:
                    char_data[key.lower()] = val

        result.update(char_data)

    # Catégories
    posted_in = soup.find("span", class_="posted_in")
    if posted_in:
        result["categories"] = ", ".join(a.get_text(strip=True) for a in posted_in.find_all("a"))
    else:
        result["categories"] = None

    return result


records = []
for item in data:
    print(f"Scraping : {item['text']}")
    try:
        record = scrape_champignon(item["href"])
        records.append(record)
    except Exception as e:
        print(f"  ERREUR ({item['href']}) : {e}")
    time.sleep(0.5)

df = pd.DataFrame(records)
print(f"\n{df.shape[0]} champignons récupérés, {df.shape[1]} colonnes")
df.head()


Scraping : AGARIC AUGUSTE
Scraping : AGARIC DES JACHÈRES
Scraping : AGARIC JAUNISSANT
Scraping : AGARIC SYLVICOLE
Scraping : AMANITE CITRINE
Scraping : AMANITE ÉPAISSE
Scraping : AMANITE FAUVE
Scraping : AMANITE JONQUILLE
Scraping : AMANITE OVOÏDE
Scraping : AMANITE PANTHÈRE
Scraping : AMANITE PHALLOÏDE
Scraping : AMANITE PRINTANIÈRE
Scraping : AMANITE ROUGISSANTE
Scraping : AMANITE TUE-MOUCHES
Scraping : AMANITE VAGINÉE
Scraping : AMANITE VIREUSE
Scraping : ARMILLAIRE COULEUR DE MIEL
Scraping : BOLET À BEAU PIED
Scraping : BOLET À CHAIR JAUNE
Scraping : BOLET À PIED CREUX
Scraping : BOLET À PIED ROUGE
Scraping : BOLET AMER
Scraping : BOLET APPENDICULÉ
Scraping : BOLET BAI
Scraping : BOLET BLAFARD
Scraping : BOLET CHÂTAIN
Scraping : BOLET CRAQUELÉ
Scraping : BOLET DE QUÉLET
Scraping : BOLET DES BOUVIERS
Scraping : BOLET DES CHÊNES VERTS
Scraping : BOLET ÉLÉGANT
Scraping : BOLET GRANULÉ
Scraping : BOLET INDIGOTIER
Scraping : BOLET JAUNE
Scraping : BOLET ORANGÉ DES CHÊNES
Scraping : BOLE

,url,nom,image_url,image_base64,nom_scientifique,statut,chapeau,pores,lamelles,pied,chair,odeur,saveur,habitat,saison,confusion,categories
0,https://www.guidedeschampignons.com/produit/ag...,AGARIC AUGUSTE,https://www.guidedeschampignons.com/wp-content...,"data:image/jpeg;base64,/9j/4QAYRXhpZgAASUkqAAg...",NaN,À rejeter,"5 à 25 cm, couvert de mèches brun-roux sur fon...",NaN,None,"Blanc, élancé (6 à 20 cm), légèrement en massu...","Blanche, jaunissant lentement à la coupe, rosé...",NaN,Douce,"Bois clairs de feuillus ou de conifères, lisiè...",Juillet > Octobre,NaN,"Agaric, Bois clairs, Champignons à rejeter, CH..."
1,https://www.guidedeschampignons.com/produit/ag...,AGARIC DES JACHÈRES,https://www.guidedeschampignons.com/wp-content...,"data:image/jpeg;base64,/9j/4QAYRXhpZgAASUkqAAg...",NaN,Excellent comestible,"5 à 15 cm, lisse, blanc puis jaunissant en vie...",NaN,None,"5 à 15 cm, élancé, avec un large anneau blanc ...","Blanche, ferme",NaN,Douce,"Surtout sous feuillus, clairières, lisières, p...",Juin > Novembre,NaN,"Agaric, CHAMPIGNONS DE AOÛT, CHAMPIGNONS DE JU..."
2,https://www.guidedeschampignons.com/produit/ag...,AGARIC JAUNISSANT,https://www.guidedeschampignons.com/wp-content...,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQA...",NaN,Toxique,"2 à 15 cm, blanc à grisâtre pâle, lisse, souve...",NaN,None,"3 à 15 cm, avec un bulbe net et un anneau floc...","Blanche, jaunissant fortement à la base du pied",NaN,NaN,"Prairies, jardins ou bois clairs",Mai > Novembre,NaN,"Agaric, Bois clairs, Champignon jardin, CHAMPI..."
3,https://www.guidedeschampignons.com/produit/ag...,AGARIC SYLVICOLE,https://www.guidedeschampignons.com/wp-content...,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQA...",NaN,Bon comestible,"3 à 12 cm, blanc à crème, centre plus ou moins...",NaN,None,"Blanc, en massue ou bulbeux, 2 à 15 cm, avec u...","Blanche, ferme",NaN,Douce,"Bois clairs, surtout de feuillus",Août > Octobre,Agaric jaunissant,"Agaric, Bois clairs, Champignons bons comestib..."
4,https://www.guidedeschampignons.com/produit/am...,AMANITE CITRINE,https://www.guidedeschampignons.com/wp-content...,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQA...",NaN,À rejeter,"4 à 10 cm, jaune citron pâle, avec des plaques...",NaN,None,"3 à 15 cm, jaunâtre, avec un anneau mince et a...",Blanche à jaune citron pâle,NaN,Douce,Forêts de feuillus et de conifères,Juillet > Octobre,NaN,"Amanite, Champignons à rejeter, CHAMPIGNONS DE..."


In [5]:
import os

RAW_DIR = "../../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

# CSV sans les photos
df_data = df.drop(columns=["image_base64"])
df_data.to_csv(f"{RAW_DIR}/champignons_data.csv", index=False)

# CSV nom + photo uniquement
df_images = df[["nom", "image_url", "image_base64"]]
df_images.to_csv(f"{RAW_DIR}/champignons_images.csv", index=False)

print(f"data   : {len(df_data)} lignes, {len(df_data.columns)} colonnes  → champignons_data.csv")
print(f"images : {len(df_images)} lignes, {len(df_images.columns)} colonnes → champignons_images.csv")


data   : 219 lignes, 16 colonnes  → champignons_data.csv
images : 219 lignes, 3 colonnes → champignons_images.csv
